# 04 — Landsat phenology cube

Build the annual Landsat index stack over the valley bottom, and **derive** the seasonal windows from observed greenness curves rather than assuming them.

**Reads** valley bottom from `01`, gallery patches from `03`  
**Writes** annual composite stack, observed phenology curves  
**Status** Phase 3 — skeleton, gate not yet opened

> Skeleton. Section headings and the config cell are in place; the analysis cells are deliberately empty for the group to fill in together.

## 0. Setup

In [ ]:
import os, sys

# PROJ/GDAL paths must be set before any geospatial import: the Jupyter kernel starts
# without `conda activate`, so PROJ cannot otherwise find its database.
def _find_share(name):
    for base in (sys.prefix, sys.base_prefix):
        p = os.path.join(base, "share", name)
        if os.path.isdir(p):
            return p
    return None

_proj, _gdal = _find_share("proj"), _find_share("gdal")
if _proj:
    os.environ["PROJ_DATA"] = os.environ["PROJ_LIB"] = _proj
if _gdal:
    os.environ.setdefault("GDAL_DATA", _gdal)

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

import pystac_client, planetary_computer
import rasterio

def _repo_root():
    """Walk up from the working directory to the repo root."""
    here = Path.cwd().resolve()
    for p in (here, *here.parents):
        if (p / ".git").exists() or (p / "environment.yml").is_file():
            return p
    raise RuntimeError("Could not find the repo root from " + str(here))

REPO     = _repo_root()
DATA_DIR = REPO / "data"
RUNS_DIR = REPO / "runs"          # manifests live here because data/ is gitignored
RUNS_DIR.mkdir(parents=True, exist_ok=True)

# ---- Figures ----
# figures/ is gitignored, but tracked via .gitkeep so it exists on a fresh clone —
# nothing to set up after cloning. Contents stay out of git because the repo is
# public and an executed run embeds a 60 cm gallery map (research plan §11).
FIG_DIR = REPO / "figures"
FIG_DPI = 300

def savefig(name, dpi=FIG_DPI):
    """Save the current figure to figures/<FIG_SUBDIR>/<name>.png.

    Call this BEFORE plt.show(): showing a figure can clear it, and you would
    silently save a blank page. FIG_SUBDIR is set in the config cell.
    """
    out = FIG_DIR / FIG_SUBDIR
    out.mkdir(parents=True, exist_ok=True)
    path = out / f"{name}.png"
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    print(f"figure -> {path.relative_to(REPO)}")
    return path

print("Imports OK")
print(f"  repo : {REPO}")

## 1. Configuration

Every parameter lives here. Pointing this notebook at another tile or another river is a single-cell edit.

In [ ]:
# ---- The pilot tile ----
TILE = "13TFJ"                       # holds Angostura, Buffalo Gap, Red Shirt, Scenic

# ---- Example reaches: one window per 8-digit USGS gauge inside 13TFJ ----
# Gauge-anchored so every window has a flow record to read alongside it (notebook 07).
# These are the walkthrough reaches, NOT the full corridor -- scaling is Phase 6.
EXAMPLE_WINDOWS = [
    {"site_no": "06401500", "name": "Angostura",   "lon": -103.4340, "lat": 43.3470},
    {"site_no": "06402600", "name": "Buffalo Gap", "lon": -103.2350, "lat": 43.4230},
    {"site_no": "06403700", "name": "Red Shirt",   "lon": -102.8921, "lat": 43.6724},
    {"site_no": "06408650", "name": "Scenic",      "lon": -102.5500, "lat": 43.7800},
]
WINDOW_HALF_M = 1000                 # half-width -> 2 x 2 km windows, as in notebook 03
# VERIFY: lon/lat for all but Red Shirt are approximate -- replace from the
# usgs_gauges layer of cheyenne_corridor_aoi.gpkg on first run.

# ---- Upstream runs ----
VBET_RUN  = "vbet_13TFJ"     # -> the 13TFJ run when Phase 1b lands
LABEL_RUN = "labels_smoketest_redshirt_06403700_2022"

# ---- Landsat ----
STAC_URL    = "https://planetarycomputer.microsoft.com/api/stac/v1"
COLLECTION  = "landsat-c2-l2"
YEAR_START  = 1984
YEAR_END    = 2025
MAX_CLOUD   = 60                     # scene-level; per-pixel QA does the real work
INDICES     = ["ndvi", "ndmi"]

# ---- Phenology windows ----
# DO NOT hardcode these. Section 5 derives them from observed curves; the plan's
# expectation (cool-season grass cures by midsummer, cottonwood stays green into
# September) is the hypothesis under test, not an input.
SPRING_WINDOW = None                 # (start_doy, end_doy), set from section 5
LATE_WINDOW   = None

SEED = 42

# ---- Outputs ----
RUN_NAME = f"phenology_{VBET_RUN}"
FIG_SUBDIR = RUN_NAME               # figures/<run>/
OUT_DIR  = DATA_DIR / RUN_NAME

## 2. Valley bottom and sample units

Clip to the valley bottom. Gallery patches from `03` are the units the curves come from.

## 3. Search and mask

Landsat C2 L2 over the window. Apply the per-pixel QA_PIXEL cloud/shadow mask before compositing — scene-level cloud cover is only a coarse prefilter.

## 4. Index stack

NDVI and NDMI per scene, stacked by date.

## 5. Observed greenness curves — the phenology bet

Mean NDVI by day-of-year for gallery patches vs. the surrounding matrix, pooled across years. **The gate question:** is there a late-season window where cottonwood is green against a cured brown matrix? HLS earns its place here — 2–3 day revisit pins the dates down far better than Landsat's 16. Set `SPRING_WINDOW` / `LATE_WINDOW` from what this shows, then re-run section 6.

## 6. Seasonal composites

Per-year composites for the derived windows, plus the spring-minus-late difference feature that should carry most of the signal.

## 7. Save and record the run

Every output gets a manifest in `runs/` — small, text, always committed, even when the raster it describes is not.

In [ ]:
manifest = {
    "run_name":    RUN_NAME,
    "notebook":    "04_Landsat_Phenology_Cube.ipynb",
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "inputs":      {},          # STAC item IDs, upstream run names, source manifests
    "parameters":  {},          # everything from the config cell
    "environment": {"python": sys.version.split()[0]},
    "results":     {},
    "outputs":     [],
}

# manifest_path = RUNS_DIR / f"{RUN_NAME}.manifest.json"
# manifest_path.write_text(json.dumps(manifest, indent=2, default=str) + "\n")

## What comes next

Gate: is there a real late-season separability signal at 30 m? If not, the 30 m arm of the design needs rethinking before notebook 05 — say so rather than pushing on.